# 02 · Mapping Fishing Activity Across Denmark (2021–2026)

**Team Vega · SODAS Hackathon 2026 · University of Copenhagen**

With the fishing-only files from notebook 01, we now visualise **where** Danish fishing
happens and **how it changes year to year**.

The key idea is a simple but effective *behavioural filter*: a vessel that is actively
**trawling** moves slowly and steadily. So we keep pings where:

- `Navigational status == "Engaged in fishing"`, **and**
- Speed over ground (`SOG`) is between **1 and 5 knots** (typical trawling speed).

We then render the result three ways:

1. **PyDeck heatmaps** — one interactive HTML map per year.
2. **Plotly density map** — a quick weighted-density view.
3. **Per-vessel route maps** — individual named vessels' tracks over a year.

In [ ]:
import os

import polars as pl
import plotly.express as px
import pydeck as pdk

os.environ["POLARS_MAX_THREADS"] = "4"

FISHING_DIR = "../data/fishing"
DOCS_DIR = "../docs"
YEARS = [2021, 2022, 2023, 2024, 2025, 2026]

## 1. Load fishing pings and apply the trawling-speed filter

We reuse the same lazy-scan pattern: filter first, collect only the slim result. The helper
below returns a tidy DataFrame for any year.

In [ ]:
def load_fishing(year: int) -> pl.DataFrame:
    """Load one year of fishing pings, keeping only active-trawling behaviour."""
    return (
        pl.scan_parquet(f"{FISHING_DIR}/fishing_aisdk-{year}-1h.parquet")
        .filter(
            (pl.col("SOG") >= 1)
            & (pl.col("SOG") <= 5)
            & (pl.col("Navigational status") == "Engaged in fishing")
        )
        .select([
        "MMSI",
        "# Timestamp",
        "Latitude",
        "Longitude",
        "Name",
        "Destination",
        "ETA",
        "Navigational status",
        "SOG",
    ])
        .collect()
    )


fishing_by_year = {year: load_fishing(year) for year in YEARS}
for year, df in fishing_by_year.items():
    print(f"{year}: {df.height:,} active-fishing pings")

## 2. Interactive heatmaps with PyDeck

For each year we build a `HeatmapLayer` centred on Danish waters and export it to a
standalone HTML file in `../docs/`. Brighter areas = more fishing activity.

The README shows static PNG snapshots of these maps; a sample interactive map
(`docs/fishing_heatmap_2025_sample.html`) is included in the repo.

In [ ]:
def build_heatmap(year: int) -> str:
    """Render one year's fishing heatmap to an HTML file and return the path."""
    points = fishing_by_year[year].select(["Latitude", "Longitude"]).to_pandas()

    layer = pdk.Layer(
        "HeatmapLayer",
        points,
        get_position="[Longitude, Latitude]",
        aggregation=pdk.types.String("SUM"),
        radius_pixels=20,
    )
    view_state = pdk.ViewState(latitude=55.0, longitude=12.0, zoom=6, pitch=0)
    deck = pdk.Deck(layers=[layer], initial_view_state=view_state)

    out_path = f"{DOCS_DIR}/fishing_heatmap_{year}.html"
    deck.to_html(out_path)
    return out_path


for year in YEARS:
    print("saved", build_heatmap(year))

## 3. Quick density view with Plotly

A lightweight alternative to PyDeck. Here we weight each point by `SOG`, plotted on an
OpenStreetMap basemap centred on Denmark. Handy for a fast look inside the notebook.

In [ ]:
df_2024 = fishing_by_year[2024].to_pandas()

fig = px.density_map(
    df_2024,
    lat="Latitude",
    lon="Longitude",
    z="SOG",                            # weight density by speed
    radius=1,
    zoom=6,
    center=dict(lat=56.5, lon=11.5),    # centred on Denmark
    map_style="open-street-map",
)
fig.update_layout(title="Fishing activity density — 2024")
fig.show()

## 4. Individual vessel routes

Zooming from the fleet to single boats: we connect each vessel's pings in time order to
draw its track. Colouring by vessel name reveals distinct fishing grounds and home ports
(see `docs/images/routes/` for exported examples like *CHRISTINA MICHELLE* and *O91 ALBATROS*).

In [ ]:
# Pick the busiest vessels in a given year so the map stays readable.
def plot_vessel_routes(year: int, top_n: int = 12):
    df = fishing_by_year[year]
    top_vessels = (
        df.group_by("Name").len().sort("len", descending=True).head(top_n)["Name"].to_list()
    )

    routes = (
        df.filter(pl.col("Name").is_in(top_vessels))
        .sort(["Name", "# Timestamp"])
        .to_pandas()
    )

    fig = px.line_map(
        routes,
        lat="Latitude",
        lon="Longitude",
        color="Name",
        zoom=6,
        center=dict(lat=56.5, lon=11.5),
        map_style="open-street-map",
    )
    fig.update_layout(title=f"Top {top_n} fishing-vessel routes — {year}")
    return fig


plot_vessel_routes(2026).show()

---
**Next:** [`03_protected_zone_analysis.ipynb`](03_protected_zone_analysis.ipynb) overlays this
fishing activity on Denmark's protected Natura 2000 marine zones to find vessels fishing
where they arguably shouldn't be.